# TURF analysis on MovieLens (ml-latest-small)

**TURF = Total Unduplicated Reach and Frequency.** It answers: *if I can only offer N items, which combination reaches the most unique people?*

- **Reach**: % of users who are "hit" by at least one item in the bundle
- **Frequency**: among reached users, the average number of items in the bundle that hit them

**Framing used here:** imagine a streaming service launching with a limited number of genre channels. Which set of genres would appeal to the most users?

- A user is **reached by a genre** if they *liked* (rated ≥ 4.0) at least `MIN_LIKES` movies in that genre.
- Part 2 repeats the analysis at the movie level, using the most-rated titles.

## 1. Setup

Download `ml-latest-small.zip` from the GroupLens page, unzip it, and put the `ml-latest-small` folder next to this notebook (or change `DATA_DIR` below).

In [ ]:
import pandas as pd
import numpy as np
from itertools import combinations
import matplotlib.pyplot as plt

DATA_DIR = "ml-latest-small"   # path to the unzipped folder

ratings = pd.read_csv(f"{DATA_DIR}/ratings.csv")
movies  = pd.read_csv(f"{DATA_DIR}/movies.csv")

print("ratings:", ratings.shape, "| movies:", movies.shape)
print("unique users:", ratings["userId"].nunique())
print("unique movies rated:", ratings["movieId"].nunique())
ratings.head()

In [ ]:
# Quick look at how people rate, to sanity-check the "liked" threshold
ratings["rating"].value_counts().sort_index().plot(kind="bar", title="Rating distribution")
plt.xlabel("rating"); plt.ylabel("count"); plt.show()

## 2. Define "reach"

These parameters drive the whole analysis, so they're worth discussing in your write-up.
If reach hits ~100% with just one or two genres, raise `MIN_LIKES` to make "reached" stricter.

In [ ]:
LIKE_THRESHOLD = 4.0   # a rating at or above this counts as "liked"
MIN_LIKES      = 3     # liked movies needed in a genre for that genre to "reach" a user
EXCLUDE        = {"(no genres listed)", "IMAX"}   # not real genres

In [ ]:
likes = ratings[ratings["rating"] >= LIKE_THRESHOLD]

# One row per (movie, genre)
movie_genres = (
    movies.assign(genre=movies["genres"].str.split("|"))
          .explode("genre")
          .loc[lambda d: ~d["genre"].isin(EXCLUDE), ["movieId", "genre"]]
)

# Count liked movies per user per genre
user_genre_likes = (
    likes.merge(movie_genres, on="movieId")
         .groupby(["userId", "genre"]).size()
         .unstack(fill_value=0)
)

# Keep every user in the base, including those who liked nothing
all_users = np.sort(ratings["userId"].unique())
user_genre_likes = user_genre_likes.reindex(all_users, fill_value=0)

# Binary reach matrix: True = this genre reaches this user
reach_matrix = user_genre_likes >= MIN_LIKES
print("users x genres:", reach_matrix.shape)
reach_matrix.head()

## 3. Reach of each genre on its own

In [ ]:
single = (reach_matrix.mean() * 100).sort_values(ascending=False).round(1)
single.to_frame("reach_%")

In [ ]:
single.sort_values().plot(kind="barh", figsize=(7, 6), title="Reach by individual genre")
plt.xlabel("% of users reached"); plt.show()

## 4. TURF: best genre bundles of each size

This checks **every** combination (exhaustive TURF). With ~18 genres, bundles up to size 5 take only a few seconds.

In [ ]:
M = reach_matrix.to_numpy()
genres = reach_matrix.columns.to_numpy()

def turf(M, labels, k):
    """Score every k-item combination by reach, then frequency."""
    rows = []
    for combo in combinations(range(M.shape[1]), k):
        sub = M[:, list(combo)]
        hit = sub.any(axis=1)
        reach = hit.mean()
        freq = sub.sum(axis=1)[hit].mean() if hit.any() else 0.0
        rows.append((" + ".join(labels[list(combo)]), reach, freq))
    out = pd.DataFrame(rows, columns=["bundle", "reach", "avg_frequency"])
    return out.sort_values(["reach", "avg_frequency"], ascending=False).reset_index(drop=True)

MAX_SIZE = 5
results = {}
for k in range(1, MAX_SIZE + 1):
    results[k] = turf(M, genres, k)
    print(f"\nTop 5 bundles of size {k}:")
    display(results[k].head(5).style.format({"reach": "{:.1%}", "avg_frequency": "{:.2f}"}))

In [ ]:
best = pd.DataFrame([
    {"size": k, "bundle": results[k].loc[0, "bundle"],
     "reach": results[k].loc[0, "reach"], "avg_frequency": results[k].loc[0, "avg_frequency"]}
    for k in results
])
display(best.style.format({"reach": "{:.1%}", "avg_frequency": "{:.2f}"}))

plt.plot(best["size"], best["reach"] * 100, marker="o")
plt.title("Maximum reach by bundle size")
plt.xlabel("number of genres in bundle"); plt.ylabel("% of users reached")
plt.xticks(best["size"]); plt.ylim(0, 105); plt.grid(alpha=0.3); plt.show()

**How to read this:** look for where the curve flattens. Past that point, adding genres barely adds new people, so the extra genres mostly deepen engagement (frequency) for people you already reach.

Also check the "Top 5" tables: when several bundles tie or nearly tie on reach, frequency is the tiebreaker, and that's often a more interesting story than the single winner.

## 5. Greedy TURF (the classic "reach build-up" chart)

Start with the highest-reach genre, then repeatedly add whichever genre brings in the most *new* users. It's fast and makes a clear presentation chart, though it isn't guaranteed to match the exhaustive best at every size.

In [ ]:
def greedy_turf(M, labels, steps):
    chosen, covered, rows = [], np.zeros(M.shape[0], dtype=bool), []
    n = M.shape[0]
    for _ in range(steps):
        gains = [(-1 if j in chosen else (~covered & M[:, j]).sum()) for j in range(M.shape[1])]
        j = int(np.argmax(gains))
        chosen.append(j)
        covered |= M[:, j]
        rows.append({"step": len(chosen), "added": labels[j],
                     "incremental_reach": gains[j] / n, "cumulative_reach": covered.mean()})
    return pd.DataFrame(rows)

greedy = greedy_turf(M, genres, steps=8)
display(greedy.style.format({"incremental_reach": "{:.1%}", "cumulative_reach": "{:.1%}"}))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(greedy["added"], greedy["incremental_reach"] * 100, label="new users added")
ax.plot(greedy["added"], greedy["cumulative_reach"] * 100, marker="o", color="black", label="cumulative reach")
ax.set_ylabel("% of users"); ax.set_ylim(0, 105)
ax.set_title("Greedy TURF: reach build-up by genre")
plt.xticks(rotation=45, ha="right"); ax.legend(); plt.tight_layout(); plt.show()

## 6. Movie-level TURF (optional)

Same idea with individual titles: from the most-rated movies, which small set is liked by the most users? There are too many combinations to check exhaustively, so this uses the greedy method.

In [ ]:
TOP_N = 50   # candidate pool: the N most-rated movies

top_movies = ratings["movieId"].value_counts().head(TOP_N).index
liked_top = likes[likes["movieId"].isin(top_movies)]

movie_matrix = (
    pd.crosstab(liked_top["userId"], liked_top["movieId"])
      .reindex(index=all_users, columns=top_movies, fill_value=0) > 0
)
titles = movies.set_index("movieId").loc[top_movies, "title"].to_numpy()

movie_greedy = greedy_turf(movie_matrix.to_numpy(), titles, steps=10)
display(movie_greedy.style.format({"incremental_reach": "{:.1%}", "cumulative_reach": "{:.1%}"}))

## 7. Caveats to mention in the write-up

- **"Liked" stands in for appeal.** A rating ≥ 4 shows a user saw and enjoyed a movie. It isn't a stated preference or purchase intent like the survey data TURF traditionally uses.
- **Sample bias.** MovieLens users are self-selected movie enthusiasts, and every user in this dataset rated at least 20 movies. Heavy raters are more likely to be "reached" by everything.
- **Genres overlap.** Many movies have several genre tags, so genre reach isn't fully independent.
- **Parameter sensitivity.** Results depend on `LIKE_THRESHOLD` and `MIN_LIKES`. Rerunning with a couple of values and showing the winners hold steady (or don't) strengthens the analysis.
- **Dataset version.** GroupLens notes the "latest" datasets change over time and aren't intended for reporting research results. That's fine for a portfolio piece, but worth a footnote.